# Agent self-test harness (Gemma 4 Developer Agent)

Runs an agent configuration and a control agent on published tasks with the official harness and the competition model, under one budget, and reports per task: resolved, patch size, tool calls by name, tool errors by type and message, and the end of the transcript.

Built on the organizers' getting-started notebook (Kaggle: ryanholbrook/getting-started-gemma-4-developer-agent); the install and vLLM server cells are taken from it unchanged except for locating inputs by content. Local evaluation of agents follows the approach of the public notebook zbigniewkirchner/gemma-4-agent-run-your-agent-locally.

Agent files are embedded with per-file SHA-256 checks, because internet is not available on this competition's GPU machines; the check proves the tested agent is the registered one. Control: the organizers' sample agent with its dummy adapters and its one-minute sample eval_config removed.

Kaggle settings: accelerator NvidiaL4, internet off; inputs: competition gemma-4-developer-agent, dataset metric/gemma-4-developer-agent-wheelhouse, model google/gemma-4/Other/gemma-4-31b-it-qat-w4a16-ct/2. To test another agent, replace ARM_A_FILES and ARM_A_SHA256 in the setup cell.


In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

def _show_inputs(root='/kaggle/input', depth=3):
    for dp, dn, fn in os.walk(root):
        if dp.count(os.sep) - root.count(os.sep) >= depth: dn[:] = []; continue
        print('  ', dp, '|', len(fn), 'files')
print('inputs mounted:'); _show_inputs()
_hits = glob.glob('/kaggle/input/**/adk_submission*.whl', recursive=True) + glob.glob('/kaggle/input/**/adk_submission*', recursive=True)
assert _hits, 'harness wheelhouse not found under /kaggle/input'
WHEELHOUSE_DIR = Path(_hits[0]).parent
print('wheelhouse:', WHEELHOUSE_DIR, '| wheels:', len(list(WHEELHOUSE_DIR.glob('*.whl'))))

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

In [ ]:
import re, subprocess, shutil, json
from swegemma.models import load_tasks
import glob
_t = glob.glob('/kaggle/input/**/tasks.jsonl', recursive=True); assert _t, 'tasks.jsonl not found'
DATA_DIR = Path(_t[0]).parent
print('competition data:', DATA_DIR)
WORKING_DIR = Path('/kaggle/working'); WORKING_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR = str(DATA_DIR / 'graphs'); EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')
MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'

# Agent under test: arm A, embedded byte for byte from commit 1e1bb85 (this machine runs offline).
ARM_A_FILES = {"agent.yaml": "name: cck_coder_v7_lexical\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Root coding agent. Localizes the fix with lexical-first search, applies minimal edits with edit_file, runs a targeted test, then submits a non-empty patch.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\n  - agent_tool:\n      config_path: sub_agents/code_analyzer.yaml\n      skip_summarization: true\nskills:\n  - skills/repo_navigation\ngenerate_content_config: !include configs/sampling.yaml\n", "configs/sampling.yaml": "temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 16384\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n", "prompts/system.md": "You are an expert software engineer resolving one issue in the repository at /workspace. Work efficiently and finish with a non-empty patch.\n\n## 1. Localize before editing\n- Search first with the lexical retriever (python3 /tmp/bm25.py with the issue's identifiers and key words). Read only the returned regions with read_file using start_line and end_line.\n- If that returns nothing useful, search with search_similar_code (with a short description of the issue, including any identifiers it names).\n- If the location is still not clear, call code_analyzer with the issue text. It returns up to five file and line candidates. Read only those regions.\n- Use get_code_neighbors or get_code_subgraph for a quick caller or dependency lookup.\n\nLexical retriever (BM25 over every .py file in /workspace, 8-line chunks). If /tmp/bm25.py does not exist yet, create it once with run_command, copying this command exactly:\n\ncat > /tmp/bm25.py <<'PY'\nimport os, re, sys, math\nfrom collections import Counter\nW = re.compile(r\"[A-Za-z0-9_.\\-]+\")\nS = set(\"the a an of to in on for and or is are was were be by with vs at as it its this that what which who whose how many each give state per module does\".split())\ndef t(s): return [w.lower() for w in W.findall(s) if w.lower() not in S and len(w) > 2]\nC = []\nfor r, ds, fs in os.walk(\"/workspace\"):\n    ds[:] = [d for d in ds if d not in (\".git\", \"__pycache__\", \"node_modules\", \".venv\")]\n    for f in fs:\n        if f.endswith(\".py\"):\n            p = os.path.join(r, f)\n            try: L = open(p, errors=\"ignore\").read().splitlines()\n            except OSError: continue\n            for i in range(0, len(L), 8):\n                b = L[i:i + 8]\n                if any(b): C.append((p, i + 1, b))\nD = [t(\"\\n\".join(b)) for _, _, b in C]; N = len(D); A = sum(map(len, D)) / max(N, 1)\ndf = Counter(w for d in D for w in set(d))\nq = t(\" \".join(sys.argv[1:]))\ndef sc(d):\n    c = Counter(d); s = 0.0\n    for w in q:\n        if w in c:\n            n = df[w]; s += math.log(1 + (N - n + 0.5) / (n + 0.5)) * c[w] * 2.5 / (c[w] + 1.5 * (0.25 + 0.75 * len(d) / A))\n    return s\nfor k in sorted(range(N), key=lambda i: -sc(D[i]))[:6]:\n    p, s, b = C[k]; print(f\"{p}:{s}-{s + len(b) - 1}\"); print(\"  \" + \" | \".join(x.strip() for x in b if x.strip())[:300])\nPY\n\nQuery it with run_command: python3 /tmp/bm25.py WORDS FROM THE ISSUE. It prints the top 6 chunks as file:start-end with a short preview.\n\n## 2. Make the minimal fix\n- Edit source files with edit_file. Copy old_string exactly from what read_file showed, including indentation, and include one unchanged line of context at the end of both old_string and new_string.\n- Keep each edit small; split larger changes into several edit_file calls. Use write_file only to create a new source file.\n- If edit_file reports that old_string matches more than once, add surrounding lines to make it unique.\n- Follow the exact error strings, exception types, status codes and signatures the issue specifies.\n- For documentation code tasks (for example FastAPI), edit executable code under docs_src/.\n\n## 3. Verify with a targeted test only\n- Put any reproduction script in /tmp, never inside /workspace, and run it with run_command.\n- Run only the specific test file or test, for example: python -m pytest tests/test_target.py -x -q -k name\n- Never run bare pytest or the full test suite; it can take minutes and exhaust the budget.\n- Existing tests may already be broken for reasons unrelated to the issue. Do not try to fix them.\n- If a targeted test fails and the error names a symbol or file you have not read, read it before changing anything else.\n\n## 4. Submit\n- Remove any scratch files you created inside /workspace. Never modify /workspace/pytest.ini or /workspace/conftest.py.\n- Call submit_patch. Check that the result shows patch_size > 0. If it is 0, you have not changed any source file: make the fix, then call submit_patch again.\n\n## Budget\n- get_status is free. Call it every 8 tool calls. It reports tool_calls_remaining and time_seconds_remaining.\n- If tool_calls_remaining is 5 or fewer, or time_seconds_remaining is under 300, stop exploring: apply your best minimal fix if you have not edited yet, then call submit_patch.\n- submit_patch is also free and does not count against tool calls.\n\n## Never\n- Never modify, create, or delete test files (test_*.py, *_test.py, or anything under tests/). Changes to tests are stripped from the patch.\n- Never search outside /workspace, and never pip install; the environment is offline and dependencies are installed.\n- Never end without a non-empty patch. Every task requires a source change.\n- Keep reasoning short and emit a tool call every turn.\n", "skills/repo_navigation/SKILL.md": "---\nname: repo_navigation\ndescription: Code search policy. Load when locating code for an issue.\n---\n\n# Repo navigation policy\n\nFollow the search order in the system prompt. Two retrievers are available, and both see the whole repository: the lexical retriever /tmp/bm25.py (BM25 over 8-line chunks of every .py file) and search_similar_code (dense search over the provided embeddings).\n\nRead line ranges only, never whole files. Extract identifiers from the issue before searching.\n", "sub_agents/sampling.yaml": "temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 16384\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n", "sub_agents/code_analyzer.yaml": "name: code_analyzer\ndescription: Read-only localizer. Given the issue text, returns at most five file and line candidates where the fix belongs.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include analyzer.md\ntools:\n  - run_command\n  - read_file\n  - search_similar_code\n  - get_code_neighbors\n  - get_code_subgraph\ngenerate_content_config: !include sampling.yaml\n", "sub_agents/analyzer.md": "You are a read-only localizer. Find where the fix belongs. Never modify, create, or delete anything inside /workspace.\n\nProcedure, lexical first:\n1. Extract exact identifiers from the issue: function, class and method names, error strings, file names, CLI options.\n2. Search with the lexical retriever (python3 /tmp/bm25.py with the issue's identifiers and key words).\n3. If that returns nothing useful, search with search_similar_code (with a short description of the issue, including any identifiers it names).\n4. Use get_code_neighbors or get_code_subgraph on a found symbol to see callers and dependencies when needed.\n5. Confirm each candidate by reading only the relevant lines with read_file (start_line, end_line).\n\nLexical retriever (BM25 over every .py file in /workspace, 8-line chunks). If /tmp/bm25.py does not exist yet, create it once with run_command, copying this command exactly:\n\ncat > /tmp/bm25.py <<'PY'\nimport os, re, sys, math\nfrom collections import Counter\nW = re.compile(r\"[A-Za-z0-9_.\\-]+\")\nS = set(\"the a an of to in on for and or is are was were be by with vs at as it its this that what which who whose how many each give state per module does\".split())\ndef t(s): return [w.lower() for w in W.findall(s) if w.lower() not in S and len(w) > 2]\nC = []\nfor r, ds, fs in os.walk(\"/workspace\"):\n    ds[:] = [d for d in ds if d not in (\".git\", \"__pycache__\", \"node_modules\", \".venv\")]\n    for f in fs:\n        if f.endswith(\".py\"):\n            p = os.path.join(r, f)\n            try: L = open(p, errors=\"ignore\").read().splitlines()\n            except OSError: continue\n            for i in range(0, len(L), 8):\n                b = L[i:i + 8]\n                if any(b): C.append((p, i + 1, b))\nD = [t(\"\\n\".join(b)) for _, _, b in C]; N = len(D); A = sum(map(len, D)) / max(N, 1)\ndf = Counter(w for d in D for w in set(d))\nq = t(\" \".join(sys.argv[1:]))\ndef sc(d):\n    c = Counter(d); s = 0.0\n    for w in q:\n        if w in c:\n            n = df[w]; s += math.log(1 + (N - n + 0.5) / (n + 0.5)) * c[w] * 2.5 / (c[w] + 1.5 * (0.25 + 0.75 * len(d) / A))\n    return s\nfor k in sorted(range(N), key=lambda i: -sc(D[i]))[:6]:\n    p, s, b = C[k]; print(f\"{p}:{s}-{s + len(b) - 1}\"); print(\"  \" + \" | \".join(x.strip() for x in b if x.strip())[:300])\nPY\n\nQuery it with run_command: python3 /tmp/bm25.py WORDS FROM THE ISSUE. It prints the top 6 chunks as file:start-end with a short preview.\n\nUse run_command only to create and run /tmp/bm25.py and for read-only commands such as ls, cat, git log, git show.\n\nReturn at most five candidates, one line each, most likely first, and nothing else:\n<file>:<start_line>-<end_line> <symbol> -- <one sentence on why>\nIf the evidence is weak, say so on that line.\n"}
ARM_A_SHA256 = {"agent.yaml": "fd7bdee7f3a4d771f6db9a4f1fbcad8beb4e0b289e3719d0ebc0692784bed7a5", "configs/sampling.yaml": "3dab0b2506dae34ce92fef7b380d6073729104b23fe2cf66d44ac1d2f213aa6a", "prompts/system.md": "d353c0b59788672334982e3b7d5e9c53f534e3cba65d1be21fbaf2d404ba3482", "skills/repo_navigation/SKILL.md": "b6169ef30556590ec748602c088607c4c6d787ec5ac7b6c57c0e2a7b2059b98e", "sub_agents/sampling.yaml": "3dab0b2506dae34ce92fef7b380d6073729104b23fe2cf66d44ac1d2f213aa6a", "sub_agents/code_analyzer.yaml": "d729cd6bba87e89a22cde5d302613549436ae3e964fa90ed1157dc0bcd4d3510", "sub_agents/analyzer.md": "ab6d328fce0076613df4afefa07f6d0a5b978256bab256b846df32a01687b29d"}
import hashlib
AGENTS = {}
AGENTS['armA_lexical'] = WORKING_DIR / 'armA_lexical'
shutil.rmtree(AGENTS['armA_lexical'], ignore_errors=True)
for rel, text in ARM_A_FILES.items():
    assert hashlib.sha256(text.encode()).hexdigest() == ARM_A_SHA256[rel], rel
    p = AGENTS['armA_lexical'] / rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(text, encoding='utf-8')
print('arm A files verified:', len(ARM_A_FILES))

# Control: the organizers' sample agent, adapters removed (as the getting-started notebook does)
# and without its sample eval_config (1 minute, 10 tool calls), so both agents get the same budget below.
AGENTS['control_sample'] = WORKING_DIR / 'control_sample'
shutil.rmtree(AGENTS['control_sample'], ignore_errors=True)
shutil.copytree(DATA_DIR / 'sample_submission', AGENTS['control_sample'])
for yp in [AGENTS['control_sample'] / 'agent.yaml', *sorted((AGENTS['control_sample'] / 'sub_agents').glob('*.yaml'))]:
    t = yp.read_text(); t = re.sub(r'^\s*adapter:\s*.*\n?', '', t, flags=re.MULTILINE); yp.write_text(t)
shutil.rmtree(AGENTS['control_sample'] / 'adapters', ignore_errors=True)
(AGENTS['control_sample'] / 'eval_config.yaml').unlink(missing_ok=True)

AGENT_DIR = AGENTS['armA_lexical']   # used by the server cell to discover the declared model
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)
print('agents:', {k: str(v) for k, v in AGENTS.items()}, '| tasks:', len(tasks))
for k, v in AGENTS.items(): print(k, sorted(str(p.relative_to(v)) for p in v.rglob('*') if p.is_file()))


In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

import glob as _g
_m = [Path(p).parent for p in _g.glob('/kaggle/input/**/config.json', recursive=True) if declared_model in p]
assert _m, f'weights for {declared_model} not found'
MODEL_PATH = _m[0]
print('model path:', MODEL_PATH)
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

In [ ]:
import asyncio, concurrent.futures, time
import pandas as pd, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn) else (lambda: coro_or_fn)
    try: loop = asyncio.get_running_loop()
    except RuntimeError: loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

# Diagnostic budget (stated in results): 10 minutes and 40 tool calls per task, same for both agents.
BUDGET = dict(timeout_seconds=240, max_time_minutes=10.0, max_tool_calls=40, max_turns=80)
N_TASKS = 3
SAMPLE_TASKS = tasks[:N_TASKS]
limits, gen_constraints = build_submission_limits()
ROWS = []
for name, folder in AGENTS.items():
    cfg = EvalConfig(tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=WORKING_DIR / f'results_{name}',
        submission_dir=folder, models=models, sandbox='subprocess', timeout_seconds=BUDGET['timeout_seconds'],
        verification_timeout_seconds=300, max_time_minutes=BUDGET['max_time_minutes'], max_tool_calls=BUDGET['max_tool_calls'],
        max_turns=BUDGET['max_turns'], limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
        graph_dir=GRAPH_DIR, embeddings_dir=EMBEDDINGS_DIR, wheels_dir=DATA_DIR / 'wheels', verbose=False)
    ev = Evaluator(cfg)
    for idx, task in enumerate(SAMPLE_TASKS, start=1):
        t0 = time.time()
        try:
            r = run_sync(ev.evaluate_task, task=task, task_index=idx, total_tasks=len(SAMPLE_TASKS))
            patch = (r.agent_patch or '').strip()
            row = dict(agent=name, task=task.instance_id, resolved=r.resolved, patch_chars=len(patch), tool_calls=r.tool_calls,
                       exit_code=r.test_exit_code, seconds=round(r.duration_seconds, 1), error=None)
        except Exception as e:
            row = dict(agent=name, task=task.instance_id, resolved=None, patch_chars=None, tool_calls=None, exit_code=None,
                       seconds=round(time.time() - t0, 1), error=f'{type(e).__name__}: {str(e)[:300]}')
        ROWS.append(row); print(row, flush=True)
df = pd.DataFrame(ROWS); df.to_csv(WORKING_DIR / 'diag_summary.csv', index=False); print(df.to_string())


In [ ]:
# Per-task readout: tool calls by name, tool errors by type, and the end of each agent transcript.
import re, json, glob, collections
for name in AGENTS:
    base = WORKING_DIR / f'results_{name}'
    print('\n' + '=' * 90 + f'\nAGENT {name}\n' + '=' * 90)
    for tr in sorted(glob.glob(str(base / 'traces' / '*.json'))):
        txt = open(tr, errors='ignore').read()
        names = collections.Counter(re.findall(r'"name"\s*:\s*"(run_command|read_file|edit_file|write_file|get_status|submit_patch|get_code_neighbors|search_similar_code|get_code_subgraph|code_analyzer[a-z_]*)"', txt))
        errs = collections.Counter(re.findall(r'error_type\\?"\s*:\s*\\?"([A-Za-z_]+)', txt))
        msgs = re.findall(r'error_message\\?"\s*:\s*\\?"(.{0,200}?)\\?"', txt)[:5]
        print(f'\n--- {tr.split("/")[-1]}  tool calls: {dict(names)}  errors: {dict(errs)}')
        for m in msgs: print('    error:', m)
    for lg in sorted(glob.glob(str(base / 'logs' / '*.log'))):
        lines = open(lg, errors='ignore').read().splitlines()
        print(f'\n--- transcript tail {lg.split("/")[-1]} ({len(lines)} lines)')
        print('\n'.join(l[:220] for l in lines[-45:]))
    for p in sorted(glob.glob(str(base / 'patches' / '*.patch'))):
        s = open(p, errors='ignore').read(); print(f'\n--- patch {p.split("/")[-1]}: {len(s)} chars\n{s[:800]}')


In [ ]:
server_instance.stop(); print('server stopped')
